# Pre- vs. post-SHINE cohort comparison: similarity to earlier ground-truth data

Are the pre-SHINE and post-SHINE cohorts *equally similar to earlier similarity data* on the images we share with it?<br>
Reference data: the Kriegeskorte et al. (2008b) 92-image set. **5a** the Mur et al. (2013) behavioral similarity judgments (16 raters); **5b** the Kriegeskorte et al. (2008b) neural data (8 human IT fMRI RDMs, 4 subjects x 2 sessions, and one pooled monkey IT RDM), for **retained subjects only** (production cohort, `status == "full data"`, `task_version >= 4.0`).<br>
Companion notebooks:
- `0_sensory.ipynb` - does SHINE change the stimuli themselves?
- `1_demographics_and_behavior.ipynb` - do they behave the same?
- `2_data_quality.ipynb` - is the data equally good?
- `3_geometry.ipynb` - do they give the same similarity structure?
- `4_semantic_hierarchy.ipynb` - are they equally similar to the Kiani hierarchy?

Backend helpers live in `ground_truth.py` (external data, image matching), `reference_agreement.py` (RDM-level) and `tree_agreement.py` (tree-level).

**Overlap.** The external `.mat` data has no image IDs, so the images were matched by content (category-restricted average-hash Hamming distance, one-to-one assignment) and the matches **confirmed by eye**: 42 of the 92 images, recorded in `gt_overlap_matches.csv` (keyed on `curated_path`). The remaining 50 have no counterpart among our images (none of our Kiani-sourced images are human faces). Everything below uses only these 42 images, i.e. 861 image pairs, about 535 of which each cohort observed.

**Statistics.** Kendall tau-a is the primary statistic and Spearman rho the secondary one. Significance comes from an image-relabeling permutation test on the pairs a cohort observed; uncertainty from a participant bootstrap of the cohort's combined RDM. Cohort equality is tested by TOST (see section 2). With 42 images, read the CIs, not only the p-values.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s1_pre_vs_post_shine/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from analysis.utils.plotting import fig_equivalence_forest
from analysis.s0_build_rdms.common import load_rdm, subset_condensed
from analysis.s0_build_rdms.behavior.spam_ratings import build_behavioral_rdm
from analysis.s1_pre_vs_post_shine import ground_truth as gt
from analysis.s1_pre_vs_post_shine.demographics_and_behavior import COHORTS, COHORT_LABEL, load_retained_cohorts
from analysis.s1_pre_vs_post_shine.geometry import MDS_CACHE_DIR, parse_subject_trials
from analysis.s1_pre_vs_post_shine.reference_agreement import (
    REFERENCE_CACHE_DIR, bh_within_cohort, bootstrap_cohort_difference, bootstrap_combined_fit_multi,
    cohort_difference_equivalence, fig_agreement_with_ci, observed_cohort_fit,
    percentile_ci, split_by_cohort,
)
from analysis.s1_pre_vs_post_shine.tree_agreement import (
    MDS_DIMS, agglomerative_agreement, fig_hdbscan_agreement, hdbscan_level_agreement, manifest_parts,
    path_tree_to_linkage, perceptual_embeddings, perceptual_tree, tree_shuffle_test,
)
from analysis.utils.stats import label_shuffle_test

## 0. Load data

In [2]:
cache_dir = REPO_ROOT / "analysis" / "results" / "parsed_data"  # override if running from a worktree without this cache copied in
rdm = {c: build_behavioral_rdm(f"{c}_shine", cache_dir=cache_dir, aggregation="simple") for c in COHORTS}
km = load_rdm("sem_km")

cohorts = load_retained_cohorts(cache_dir)
subject_ids = {c: cohorts["participant_ids"][c].tolist() for c in COHORTS}
trials_by_subject, n_conds = parse_subject_trials(cohorts["df_trials"], subject_ids)
trials = split_by_cohort(trials_by_subject, subject_ids)

overlap = gt.load_overlap()
rows = overlap["manifest_row"].to_numpy()
stimuli = gt.load_stimuli()
targets = gt.external_targets(overlap)
means = {name: gt.mean_rdm(rdms) for name, rdms in targets.items()}
perceptual = {c: subset_condensed(rdm[c], rows) for c in COHORTS}
km_overlap = subset_condensed(km, rows)
joint = ~np.isnan(perceptual["pre"]) & ~np.isnan(perceptual["post"])

SEED, N_PERM, N_BOOT, N_BOOT_DIFF = 0, 5000, 500, 1000
TARGET = {"behavioral_mur2013": "Mur 2013 ratings", "hIT_kriegeskorte2008": "Human IT (fMRI)", "mIT_kriegeskorte2008": "Monkey IT"}
cache = lambda name: REFERENCE_CACHE_DIR / name
print(f"{len(overlap)} shared images, {len(km_overlap)} pairs; observed by pre {int((~np.isnan(perceptual['pre'])).sum())}, "
      f"post {int((~np.isnan(perceptual['post'])).sum())}, both {int(joint.sum())}")

[behavioral] variant='pre_shine': 79 participants.


[behavioral] Built 1738 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 64.57% covered (92977 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.95, sd=1.13.
[behavioral] WARNING: 92977 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_pre_simple.npy  (length 262450)


[behavioral] variant='post_shine': 76 participants.


[behavioral] Built 1672 per-trial partial observation sets (screening + experimental pooled).


[behavioral] Pair coverage: 63.09% covered (96882 of 262450 pairs never jointly observed). Judgements/pair (covered pairs only): mean=1.92, sd=1.11.
[behavioral] WARNING: 96882 pairs have zero observations and will be NaN in the saved RDM -- this is a real coverage gap, not silently interpolated.
[behavioral] Saved D_behav_post_simple.npy  (length 262450)


42 shared images, 861 pairs; observed by pre 536, post 534, both 334


## 1. The 42 shared images

Each tile is the 92-image dataset's image (left) next to ours, pre-SHINE (right), sorted by position in our manifest. The post-SHINE cohort saw the SHINE-processed version of the same images.

In [3]:
gt.fig_match_grid(stimuli, overlap).show()
overlap.groupby("ext_category").size().rename("n").to_frame().T

ext_category,artificial inanimate,human bodypart,natural inanimate,nonhuman bodypart,nonhuman face
n,10,5,19,2,6


## 2. Agreement with the external data, and equivalence of the cohorts

Per cohort and external target: tau-a and rho between the cohort's combined RDM (restricted to the 42 images) and the external RDM (mean over its raters/sessions); the permutation p; the participant-bootstrap CI of the combined RDM's tau-a; the mean fit of each individual external RDM; and the external **leave-one-out ceiling**, how well a single external rater/session is predicted by the others. The ceiling is a different quantity from our combined-RDM value (a single rater vs a pooled cohort), so it marks headroom rather than a strict cap. The monkey RDM is a single pooled RDM, so it has no ceiling and no per-rater spread; the 8 human-IT RDMs are 4 subjects x 2 sessions, so they are not independent.

**Equivalence (TOST).** D = tau-a(pre) - tau-a(post) per target. The cohorts are disjoint participants, so each is bootstrapped on its own and re-aggregated per draw; D must have its 90% CI inside +/- bound, where the bound is (z(0.95) + z(0.90)) x SE rounded up to 0.01 (`tost_bound_multiplier` in `analysis/utils/equivalence.py`): the smallest bound at which a true difference of zero is declared equivalent with 80% probability (section 4 checks this). There is no pre-defined smallest effect size of interest, so a bound says what the sample can resolve, not that a difference is negligible. `own` scores each cohort on the pairs it observed; `joint` only on the pairs both cohorts observed.

In [4]:
boots = {
    c: bootstrap_combined_fit_multi(
        trials[c], means, n_conds, n_boot=N_BOOT, seed=SEED, subset=rows,
        cache_path=cache(f"gt_bootstrap_{c}_n{N_BOOT}_seed{SEED}.pkl"),
    ) for c in COHORTS
}
rows_out = []
for c in COHORTS:
    for name in targets:
        permutation = label_shuffle_test(
            perceptual[c], means[name], n_perm=N_PERM, seed=SEED, cache_path=cache(f"gt_permutation_{c}_{name}_n{N_PERM}_seed{SEED}.npz"),
        )
        ci = percentile_ci(boots[c].loc[boots[c]["model"] == name, "kendall_a"])
        rows_out.append({
            "cohort": c, "target": name, "tau_a": permutation.observed["kendall_a"], "ci_lo": ci[0], "ci_hi": ci[1],
            "spearman": permutation.observed["spearman"], "permutation_p (tau-a)": permutation.p_value["kendall_a"],
            "mean_fit_per_external_rdm": gt.per_external_fit(perceptual[c], targets[name]).mean(),
            "external_leave_one_out": gt.external_leave_one_out(targets[name]),
        })
summary = pd.DataFrame(rows_out)
summary.round(4)

,cohort,target,tau_a,ci_lo,ci_hi,spearman,permutation_p (tau-a),mean_fit_per_external_rdm,external_leave_one_out
0,pre,behavioral_mur2013,0.2584,0.2025,0.2934,0.3816,0.0002,0.2044,0.3555
1,pre,hIT_kriegeskorte2008,0.1284,0.0916,0.1648,0.1947,0.0002,0.0657,0.1291
2,pre,mIT_kriegeskorte2008,0.1574,0.1207,0.1944,0.2341,0.0002,0.1574,NaN
3,post,behavioral_mur2013,0.2697,0.2044,0.2999,0.4044,0.0002,0.2358,0.3555
4,post,hIT_kriegeskorte2008,0.1500,0.0932,0.1795,0.2258,0.0002,0.0718,0.1291
5,post,mIT_kriegeskorte2008,0.1729,0.1104,0.2076,0.2610,0.0002,0.1729,NaN


In [5]:
boots_diff = bootstrap_cohort_difference(
    trials["pre"], trials["post"], means, n_conds, n_boot=N_BOOT_DIFF, seed=SEED, subset=rows, pair_masks={"own": None, "joint": joint},
    cache_path=cache(f"gt_cohort_difference_n{N_BOOT_DIFF}_seed{SEED}.pkl"),
)
observed = {
    "own": observed_cohort_fit(perceptual["pre"], perceptual["post"], means),
    "joint": observed_cohort_fit(perceptual["pre"], perceptual["post"], means, mask=joint),
}
equivalence = cohort_difference_equivalence(observed, boots_diff, len(subject_ids["pre"]), len(subject_ids["post"]))
equivalence[["pairs", "target", "statistic", "effect", "ci_low", "ci_high", "bound", "min_bound", "se", "equivalent"]].round(4)

,pairs,target,statistic,effect,ci_low,ci_high,bound,min_bound,se,equivalent
0,joint,behavioral_mur2013,kendall_a,-0.0279,-0.1018,0.0395,0.13,0.1018,0.0435,True
1,joint,behavioral_mur2013,spearman,-0.0422,-0.1446,0.0522,0.18,0.1446,0.0601,True
2,joint,hIT_kriegeskorte2008,kendall_a,-0.0053,-0.0539,0.0597,0.11,0.0597,0.0345,True
3,joint,hIT_kriegeskorte2008,spearman,-0.0044,-0.0787,0.0916,0.16,0.0916,0.0518,True
4,joint,mIT_kriegeskorte2008,kendall_a,0.0026,-0.0625,0.0669,0.12,0.0669,0.0388,True
5,joint,mIT_kriegeskorte2008,spearman,-0.0023,-0.0915,0.0922,0.17,0.0922,0.0568,True
6,own,behavioral_mur2013,kendall_a,-0.0112,-0.0632,0.0483,0.10,0.0632,0.0340,True
7,own,behavioral_mur2013,spearman,-0.0228,-0.0929,0.0633,0.14,0.0929,0.0477,True
8,own,hIT_kriegeskorte2008,kendall_a,-0.0216,-0.0580,0.0412,0.09,0.0580,0.0292,True
9,own,hIT_kriegeskorte2008,spearman,-0.0312,-0.0863,0.0608,0.13,0.0863,0.0434,True


## 2a. Mur et al. (2013) behavioral ratings

In [6]:
mur = "behavioral_mur2013"
ceiling_mur = summary.loc[summary["target"] == mur, "external_leave_one_out"].iloc[0]
fig_agreement_with_ci(summary[summary["target"] == mur], targets={mur: TARGET[mur]}, ceiling=(mur, ceiling_mur)).show()

def forest(target_names, title):
    rows = equivalence[equivalence["target"].isin(target_names)]
    return fig_equivalence_forest(
        {f"{TARGET[r.target]}, {'tau-a' if r.statistic == 'kendall_a' else 'rho'}, {r.pairs} pairs": r._asdict() for r in rows.itertuples()},
        x_title="pre - post (90% CI; grey ticks = equivalence bound)", title=title,
    )

forest([mur], "Equivalence of pre- and post-SHINE agreement with the Mur ratings").show()

### 2a.1 Do the perceptual trees match the Mur tree?

The Mur dendrogram is the average-linkage tree of the mean of the 16 raters' RDMs on the 42 images. Each cohort's side is its cached MDS embedding (all 725 images, `3_geometry.ipynb`'s fits) restricted to the 42 images, average linkage, at d = 3, 9, 15. Baker's gamma has a shuffle null that permutes image identity in the Mur tree. For context, the same is computed against the Kiani tree restricted to the 42 images, and the Kiani tree against the Mur tree.

In [7]:
embedded = {
    c: {d: subset_condensed(e, rows) for d, e in perceptual_embeddings(rdm[c], c, MDS_DIMS, MDS_CACHE_DIR).items()} for c in COHORTS
}
parts_overlap = [manifest_parts()[r] for r in rows]
z_mur = perceptual_tree(means[mur])
z_km = path_tree_to_linkage(parts_overlap)

tree_rows = []
for c in COHORTS:
    vs_mur = agglomerative_agreement(z_mur, embedded[c]).set_index("ndim")
    vs_km = agglomerative_agreement(z_km, embedded[c]).set_index("ndim")
    for d in MDS_DIMS:
        shuffle = tree_shuffle_test(
            perceptual_tree(embedded[c][d]), z_mur, n_perm=N_PERM, seed=SEED,
            cache_path=cache(f"gt_tree_shuffle_{c}_d{d}_n{N_PERM}_seed{SEED}.npz"),
        )
        tree_rows.append({
            "cohort": c, "ndim": d, "gamma_vs_mur": shuffle.observed, "null_p95": shuffle.summary()["null_p95"],
            "p_value": shuffle.p_value, "gamma_vs_kiani": vs_km.at[d, "baker_gamma"],
            "cophenetic_fidelity": vs_mur.at[d, "cophenetic_fidelity"],
        })
trees = pd.DataFrame(tree_rows)
context = tree_shuffle_test(z_km, z_mur, n_perm=N_PERM, seed=SEED, cache_path=cache(f"gt_tree_shuffle_kiani_vs_mur_n{N_PERM}_seed{SEED}.npz"))
print(f"Kiani tree vs Mur tree (42 images): gamma = {context.observed:.3f}, shuffle p = {context.p_value:.4f}")
trees.round(4)

Kiani tree vs Mur tree (42 images): gamma = 0.571, shuffle p = 0.0002


,cohort,ndim,gamma_vs_mur,null_p95,p_value,gamma_vs_kiani,cophenetic_fidelity
0,pre,3,0.6141,0.0701,0.0002,0.5952,0.7925
1,pre,9,0.7677,0.0611,0.0002,0.6767,0.6719
2,pre,15,0.6094,0.0915,0.0002,0.8854,0.6933
3,post,3,0.7236,0.0620,0.0002,0.6666,0.7915
4,post,9,0.6075,0.0916,0.0002,0.8817,0.7213
5,post,15,0.6083,0.0915,0.0002,0.8829,0.7481


## 2b. Kriegeskorte et al. (2008b) neural data

Human IT (fMRI) and monkey IT, same statistics as 2a. Human IT agreement is expected to be lower than behavior and is compared with its own leave-one-out ceiling.

In [8]:
neural = ["hIT_kriegeskorte2008", "mIT_kriegeskorte2008"]
ceiling_hit = summary.loc[summary["target"] == neural[0], "external_leave_one_out"].iloc[0]
fig_agreement_with_ci(summary[summary["target"].isin(neural)], targets={t: TARGET[t] for t in neural}, ceiling=(neural[0], ceiling_hit)).show()
forest(neural, "Equivalence of pre- and post-SHINE agreement with the Kriegeskorte neural data").show()

## 3. Multiple comparisons

Benjamini-Hochberg within each cohort over its confirmatory family: the three tau-a permutation tests (one per external target) and the three Baker's-gamma shuffle tests against the Mur tree (d = 3, 9, 15). Cohorts are never pooled. The Spearman values, bootstrap CIs and equivalence tests are not part of the family, and the three targets are not corrected against each other for the equivalence claims: each is its own claim.

In [9]:
family = pd.concat([
    summary[["cohort", "target", "permutation_p (tau-a)"]].rename(columns={"permutation_p (tau-a)": "p_value"}).assign(
        test=lambda d: "tau-a permutation, " + d["target"]),
    trees.assign(test=lambda d: "Baker gamma vs Mur tree, d = " + d["ndim"].astype(str)),
], ignore_index=True)[["cohort", "test", "p_value"]]
bh_within_cohort(family).round(4)

,cohort,test,p_value,p_fdr
0,pre,"tau-a permutation, behavioral_mur2013",0.0002,0.0002
1,pre,"tau-a permutation, hIT_kriegeskorte2008",0.0002,0.0002
2,pre,"tau-a permutation, mIT_kriegeskorte2008",0.0002,0.0002
3,post,"tau-a permutation, behavioral_mur2013",0.0002,0.0002
4,post,"tau-a permutation, hIT_kriegeskorte2008",0.0002,0.0002
5,post,"tau-a permutation, mIT_kriegeskorte2008",0.0002,0.0002
6,pre,"Baker gamma vs Mur tree, d = 3",0.0002,0.0002
7,pre,"Baker gamma vs Mur tree, d = 9",0.0002,0.0002
8,pre,"Baker gamma vs Mur tree, d = 15",0.0002,0.0002
9,post,"Baker gamma vs Mur tree, d = 3",0.0002,0.0002


## 4. Does the equivalence rule work? Calibration on random halves of one cohort

The same procedure on random halves of the pre cohort, where the true difference is zero by construction: how often does the bound rule declare equivalence? By construction the expected rate at a true difference of zero is 80% (assuming a roughly normal bootstrap distribution). Halves have about half the participants, so their SE and bound are larger than in the real comparison; this checks the rule, not the real bound. Each split draws two disjoint halves, bootstraps each (`n_boot` draws) and applies the TOST for every target and statistic.

In [10]:
calibration = gt.calibrate_equivalence_bound(
    trials["pre"], means, n_conds, rows, n_splits=10, n_boot=200, seed=100, cache_path=cache("gt_calibration_pre_s10_b200_seed100.pkl"),
)
print(f"equivalent in {int(calibration['equivalent'].sum())} of {len(calibration)} split x target x statistic tests "
      f"({calibration['equivalent'].mean():.0%}; the rule targets 80%)")
calibration.groupby(["target", "statistic"])["equivalent"].mean().unstack().round(2)

equivalent in 46 of 60 split x target x statistic tests (77%; the rule targets 80%)


statistic,kendall_a,spearman
target,,
behavioral_mur2013,0.6,0.6
hIT_kriegeskorte2008,0.9,0.9
mIT_kriegeskorte2008,0.8,0.8


## 5. Exploratory: HDBSCAN on the 42 images

At N = 42 HDBSCAN's `min_cluster_size` must be tiny and most Kiani subcategories have a handful of images, so this is exploratory only: each cohort's embedded geometry (restricted to the 42 images) against the Kiani categories at each directory level, by the adjusted Rand index over the images HDBSCAN clustered.

In [11]:
hdbscan = pd.concat(
    [hdbscan_level_agreement(embedded[c], parts_overlap, (1, 2, 3), (2, 3, 5)).assign(cohort=c) for c in COHORTS], ignore_index=True,
)
fig_hdbscan_agreement(hdbscan, level_names={1: "Level 1: domain", 2: "Level 2: mid-level", 3: "Level 3: basic"}, ndim=9).show()
hdbscan[hdbscan["ndim"] == 9].pivot_table(index=["cohort", "min_cluster_size"], columns="km_level", values=["ari", "frac_noise"]).round(2)

ari             frac_noise            
km_level                    1     2     3          1     2     3
cohort min_cluster_size                                         
post   2                 0.43  0.84  0.92       0.10  0.10  0.10
       3                 0.55  1.00  0.74       0.07  0.07  0.07
       5                 0.56  0.79  0.59       0.10  0.10  0.10
pre    2                 0.37  0.77  0.93       0.05  0.05  0.05
       3                 0.55  0.97  0.78       0.10  0.10  0.10
       5                 0.53  0.84  0.67       0.12  0.12  0.12